In [2]:
# -*- coding: utf-8 -*-

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import MNLogit
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.multitest import multipletests
from scipy.stats import chi2
import warnings
import os

warnings.filterwarnings('ignore')

# ============================================================================
# 0. 配置
# ============================================================================
output_dir = r"mediation_results_counterfactual"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

np.random.seed(2026)

# ============================================================================
# 1. 读入数据
# ============================================================================
df_val = pd.read_excel(r"gist_data2_4alldata1.xlsx")


continuous_vars = ['MaximumTumorSize', 'KI_67']
for var in continuous_vars:
    if var in df_val.columns:
        mean_val = df_val[var].mean()
        std_val = df_val[var].std()
        if std_val > 0:
            df_val[var] = (df_val[var] - mean_val) / std_val

# Variable definition 


Y = "RiskCategory_encoded"
X_list = ['EndoscopicUltrasoundType','Origin Layer','Echo Uniformity','Growth Pattern','Boundary Clarity',
          'MaximumTumorSize','Location','Ulceration','ShapeRegularity','BloodFlowSignal','Echo']  #
M_list = ['CD34','CD117','PDGFR-α','Dog-1','S-100','SMA','KI_67']


df_val['GrowthPattern_binary'] = df_val['Growth Pattern'].map({0:0, 1:0, 2:1})
df_val['EUS_type_binary'] = df_val['EndoscopicUltrasoundType'].map({1:0, 2:1, 3:1}) 

X_list.remove('Growth Pattern')
X_list.remove('EndoscopicUltrasoundType')
X_list.append('GrowthPattern_binary')
X_list.append('EUS_type_binary')



In [4]:

# ============================================================================
# proportional odds
# ============================================================================
def test_proportional_odds(df, X, M, Y):
    data = df[[X, M, Y]].dropna().copy()
    if len(data) < 20:
        return np.nan
    data[Y] = pd.Categorical(data[Y]).codes
    try:
        model_ord = OrderedModel(data[Y], data[[X, M]], distr='logit')
        res_ord = model_ord.fit(disp=False)
        ll_ord = res_ord.llf
        X_const = sm.add_constant(data[[X, M]])
        model_mn = MNLogit(data[Y], X_const)
        res_mn = model_mn.fit(disp=False)
        ll_mn = res_mn.llf
        lr = 2 * (ll_mn - ll_ord)
        df_lr = (len(data[Y].unique())-2) * 2
        p = chi2.sf(lr, df_lr)
        return p
    except:
        return np.nan

po_results = {}
for X in X_list:
    for M in M_list:
        p = test_proportional_odds(df_val, X, M, Y)
        po_results[(X, M)] = p
        print(f"{X} → {M}: p={p:.4f}")

# 将po_results转换为DataFrame
po_df = pd.DataFrame([{'X': k[0], 'M': k[1], 'p_value': v} for k, v in po_results.items()])
# 保存到指定文件夹（假设output_dir已定义）
po_df.to_csv(os.path.join(output_dir, "proportional_odds_test_results.csv"), index=False,encoding='utf-8-sig')


Origin Layer → CD34: p=0.1397
Origin Layer → CD117: p=0.0750
Origin Layer → PDGFR-α: p=0.1010
Origin Layer → Dog-1: p=0.3396
Origin Layer → S-100: p=0.4113
Origin Layer → SMA: p=0.4890
Origin Layer → KI_67: p=0.0189
Echo Uniformity → CD34: p=0.2430
Echo Uniformity → CD117: p=0.1179
Echo Uniformity → PDGFR-α: p=0.1289
Echo Uniformity → Dog-1: p=0.5024
Echo Uniformity → S-100: p=0.5521
Echo Uniformity → SMA: p=0.5735
Echo Uniformity → KI_67: p=0.0538
Boundary Clarity → CD34: p=0.0893
Boundary Clarity → CD117: p=0.0359
Boundary Clarity → PDGFR-α: p=0.0595
Boundary Clarity → Dog-1: p=0.2092
Boundary Clarity → S-100: p=0.2725
Boundary Clarity → SMA: p=0.3725
Boundary Clarity → KI_67: p=0.0405
MaximumTumorSize → CD34: p=0.0000
MaximumTumorSize → CD117: p=0.0000
MaximumTumorSize → PDGFR-α: p=0.0000
MaximumTumorSize → Dog-1: p=0.0000
MaximumTumorSize → S-100: p=0.0000
MaximumTumorSize → SMA: p=0.0000
MaximumTumorSize → KI_67: p=0.0000
Location → CD34: p=0.0424
Location → CD117: p=0.0202
Locati

In [5]:
X_list = ['EndoscopicUltrasoundType','Origin Layer','Echo Uniformity','Growth Pattern','Boundary Clarity',
          'MaximumTumorSize','Location','Ulceration','ShapeRegularity','BloodFlowSignal','Echo']  #
M_list = ['CD34','CD117','PDGFR-α','Dog-1','S-100','SMA','KI_67']



df_val['GrowthPattern_binary'] = df_val['Growth Pattern'].map({0:0, 1:0, 2:1})
df_val['EUS_type_binary'] = df_val['EndoscopicUltrasoundType'].map({1:0, 2:1, 3:1}) 

X_list.remove('Growth Pattern')
X_list.remove('EndoscopicUltrasoundType')
X_list.append('GrowthPattern_binary')
X_list.append('EUS_type_binary')


In [ ]:
np.random.seed(2026)

# ============================================================================
# 3.  stratified_bootstrap 
# ============================================================================
def stratified_bootstrap(y, B=2000):
    y = np.array(y)
    levels = np.unique(y)
    idx_list = []
    for _ in range(B):
        idx = []
        for lev in levels:
            pos = np.where(y == lev)[0]
            idx.extend(np.random.choice(pos, size=len(pos), replace=True))
        np.random.shuffle(idx)
        idx_list.append(idx)
    return idx_list

# ============================================================================
# 4. The core of counterfactual mediation analysis
# ============================================================================
def counterfactual_mediation(df, X, M, Y, B=2000):
    # 检查暴露类型
    data = df[[X, M, Y]].dropna().copy()
    if len(data) < 30:
        return None
    
  
    if X == 'MaximumTumorSize':
        # Continuous exposure
        x_high, x_low = 0.5, -0.5
    else:
        # It must be a binary classification of 0/1
        uniq = data[X].dropna().unique()
        if not (set(uniq) <= {0, 1}):
            return None
        x_high, x_low = 1, 0
    
    # 编码结局
    data[Y] = pd.Categorical(data[Y]).codes
    M_binary = data[M].nunique() == 2
    
   
    boot_idx = stratified_bootstrap(data[Y].values, B)
    
    TE_list, NIE_list, NDE_list = [], [], []
    valid_boot = 0
    
    for idx in boot_idx:
        boot = data.iloc[idx]
        try:
            if M_binary:
                m_model = sm.Logit(boot[M], sm.add_constant(boot[X])).fit(disp=False, maxiter=500)
            else:
                m_model = sm.OLS(boot[M], sm.add_constant(boot[X])).fit()

            # MNLogit
            Xb = sm.add_constant(boot[[X, M]])
            
            try:
                y_model = MNLogit(boot[Y], Xb).fit(disp=False, maxiter=1000)
            except np.linalg.LinAlgError:
                continue  

            n = len(boot)
            X1 = np.full(n, x_high)
            X0 = np.full(n, x_low)

            # predicted  value
            df_X1 = pd.DataFrame({X: X1})
            df_X1_const = sm.add_constant(df_X1, has_constant='add')
            df_X0_const = sm.add_constant(pd.DataFrame({X: X0}), has_constant='add')
            if M_binary:
                M1 = m_model.predict(df_X1_const)
                M0 = m_model.predict(df_X0_const)
            else:
                M1 = m_model.predict(df_X1_const)
                M0 = m_model.predict(df_X0_const)

            # Expectation of outcome
            def pred_Y(x_vals, m_vals):
                df_pred = pd.DataFrame({X: x_vals, M: m_vals})
                df_pred_const = sm.add_constant(df_pred, has_constant='add')
                probs = y_model.predict(df_pred_const)
                categories = np.arange(probs.shape[1])
                return (probs * categories).sum(axis=1)

            Y11 = pred_Y(X1, M1)
            Y10 = pred_Y(X1, M0)
            Y00 = pred_Y(X0, M0)

            TE = np.mean(Y11 - Y00)
            NIE = np.mean(Y11 - Y10)
            NDE = np.mean(Y10 - Y00)

            TE_list.append(TE)
            NIE_list.append(NIE)
            NDE_list.append(NDE)
            valid_boot += 1
        except Exception as e:
            # print(f"  Bootstrap error: {e}")
            continue
    
    TE_arr = np.array(TE_list)
    NIE_arr = np.array(NIE_list)
    NDE_arr = np.array(NDE_list)
    
    TE = np.mean(TE_arr)
    NIE = np.mean(NIE_arr)
    NDE = np.mean(NDE_arr)
    
 
    TE_ci = np.percentile(TE_arr, [2.5, 97.5])
    NIE_ci = np.percentile(NIE_arr, [2.5, 97.5])
    NDE_ci = np.percentile(NDE_arr, [2.5, 97.5])
    
    PM = NIE / TE if TE != 0 else np.nan
    p_boot = 2 * min(np.mean(NIE_arr <= 0), np.mean(NIE_arr >= 0))
    p_boot = min(p_boot, 1.0)
    
    return {
        'X': X, 'M': M,
        'TE': TE, 'TE_CI_low': TE_ci[0], 'TE_CI_high': TE_ci[1],
        'NIE': NIE, 'NIE_CI_low': NIE_ci[0], 'NIE_CI_high': NIE_ci[1],
        'NDE': NDE, 'NDE_CI_low': NDE_ci[0], 'NDE_CI_high': NDE_ci[1],
        'Prop_Mediated': PM, 'indirect_p': p_boot,
        'N': len(data), 'Bootstrap_N': valid_boot
    }

# ============================================================================
# 5. run
# ============================================================================
results = []
for X in X_list:
    for M in M_list:
        print(f"\n{X} → {M}")
        res = counterfactual_mediation(df_val, X, M, Y, B=2000)
        if res:
            results.append(res)
            print(f"  NIE={res['NIE']:.4f} (95% CI: {res['NIE_CI_low']:.4f}, {res['NIE_CI_high']:.4f}), p={res['indirect_p']:.4f}")
        else:
            print("  失败")

df_res = pd.DataFrame(results)
df_res.to_csv(os.path.join(output_dir, "counterfactual_results.csv"), index=False,encoding='utf-8-sig')




Origin Layer → CD34
  NIE=0.0160 (95% CI: -0.1258, 0.1432), p=1.0000

Origin Layer → CD117
  NIE=0.0098 (95% CI: -0.2196, 0.2277), p=1.0000

Origin Layer → PDGFR-α
  NIE=-0.0000 (95% CI: -0.0521, 0.0501), p=1.0000

Origin Layer → Dog-1
  NIE=-0.0528 (95% CI: -0.2253, 0.0138), p=0.6938

Origin Layer → S-100
  NIE=-0.0031 (95% CI: -0.0382, 0.0318), p=1.0000

Origin Layer → SMA
  NIE=-0.0058 (95% CI: -0.1005, 0.0969), p=1.0000

Origin Layer → KI_67
  NIE=-0.0478 (95% CI: -0.4622, 0.2491), p=1.0000

Echo Uniformity → CD34
  NIE=-0.0702 (95% CI: -0.2140, 0.1538), p=0.4753

Echo Uniformity → CD117
  NIE=-0.1725 (95% CI: -0.4343, 0.0000), p=0.5846

Echo Uniformity → PDGFR-α
  NIE=-0.0009 (95% CI: -0.0279, 0.0197), p=0.9620

Echo Uniformity → Dog-1
  NIE=0.0094 (95% CI: -0.0187, 0.0415), p=0.4389

Echo Uniformity → S-100
  NIE=0.0000 (95% CI: -0.0146, 0.0149), p=0.9865

Echo Uniformity → SMA
  NIE=-0.0325 (95% CI: -0.0804, 0.0036), p=0.0850

Echo Uniformity → KI_67
  NIE=-0.0837 (95% CI: -0.2

In [ ]:
np.random.seed(2026)

pvals = df_res['indirect_p'].dropna().values
if len(pvals) > 0:
    reject, p_fdr, _, _ = multipletests(pvals, method='fdr_bh')
    df_res.loc[df_res['indirect_p'].notna(), 'indirect_p_fdr'] = p_fdr
    df_res['significant_fdr'] = False
    df_res.loc[df_res['indirect_p'].notna(), 'significant_fdr'] = reject
    df_res.to_csv(os.path.join(output_dir, "mediation_results_fdr.csv"), index=False,encoding='utf-8-sig')
    print(f"\nFDR completed, number of significant paths: {sum(reject)}")
else:
    print("No valid p-value, skip FDR")

sig_paths = df_res[df_res['significant_fdr']][['X','M']].values.tolist() if 'significant_fdr' in df_res else []

sensitivity_results = []
for X, M in sig_paths:
 # ---------- 1.  Obtain the OR of the mediator-outcome relationship and calculate the E-value ----------
    data_b = df_val[[X, M, Y]].dropna().copy()
    data_b[Y] = pd.Categorical(data_b[Y]).codes
  
    Xb = sm.add_constant(data_b[[X, M]])
    y_model_b = MNLogit(data_b[Y], Xb).fit(disp=False, maxiter=1000)
    b_coef = y_model_b.params.iloc[:, 1].mean()   
    or_my = np.exp(b_coef)   
    # E-value
    if or_my > 1:
        e_val = or_my + np.sqrt(or_my * (or_my - 1))
    else:
        e_val = 1

    # ---------- 2. Robustness of linear regression----------
    data_lin = df_val[[X, M, Y]].dropna().copy()
    data_lin[Y] = data_lin[Y].astype(float)
    M_is_binary = data_lin[M].nunique() == 2
    np.random.seed(2026)
    boot_acme = []
    for _ in range(2000):
        idx = np.random.choice(len(data_lin), len(data_lin), replace=True)
        boot = data_lin.iloc[idx]
        try:
            if M_is_binary:
                a_b = sm.Logit(boot[M], sm.add_constant(boot[X])).fit(disp=False).params[X]
            else:
                a_b = sm.OLS(boot[M], sm.add_constant(boot[X])).fit().params[X]
            b_b = sm.OLS(boot[Y], sm.add_constant(boot[[X, M]])).fit().params[M]
            boot_acme.append(a_b * b_b)
        except:
            continue
    if len(boot_acme) >= 100:
        acme_lin = np.mean(boot_acme)
        ci_lin = np.percentile(boot_acme, [2.5, 97.5])
        lin_ci = f"{ci_lin[0]:.4f}–{ci_lin[1]:.4f}"
    else:
        acme_lin = np.nan
        lin_ci = "Failed"

    # ---------- 3. Different Bootstrap iterations (to test the stability of NIE confidence intervals)----------
    boot_cis = {}
    for n in [1000, 2000, 3000, 4000, 5000]:
        res_temp = counterfactual_mediation(df_val, X, M, Y, B=n)
        if res_temp:
            boot_cis[n] = f"{res_temp['NIE_CI_low']:.4f}–{res_temp['NIE_CI_high']:.4f}"
        else:
            boot_cis[n] = "Failed"

    sensitivity_results.append({
        'Path': f"{X}→{M}",
        'OR_M_Y': f"{or_my:.3f}",
        'E_value': f"{e_val:.2f}",
        'Linear_ACME': f"{acme_lin:.4f}" if not np.isnan(acme_lin) else "Failed",
        'Linear_95%CI': lin_ci,
        'Boot1000': boot_cis.get(1000, "Failed"),
        'Boot2000': boot_cis.get(2000, "Failed"),
        'Boot3000': boot_cis.get(3000, "Failed"),
        'Boot4000': boot_cis.get(4000, "Failed"),
        'Boot5000': boot_cis.get(5000, "Failed")
    })

df_sens = pd.DataFrame(sensitivity_results)
df_sens.to_csv(os.path.join(output_dir, "sensitivity_analysis_counterfactual.csv"), index=False,encoding='utf-8-sig')
print("\nSensitivity analysis results:\n", df_sens.to_string(index=False))